In [ ]:
# import torch
# from torch.utils.data import DataLoader, random_split, TensorDataset
# import os
# from tqdm import tqdm

# from data_loader.dataset import EuroSatDataset
# from attacks.pgd import PGD
# from config import *

# def create_pgd_test_dataset(
#     model,
#     test_dataset,
#     device,
#     epsilon=0.02,
#     alpha=0.004,
#     iterations=5,
#     batch_size=32,
#     random_start=True
# ):
#     """
#     Create a fixed PGD adversarial test dataset
#     """
#     model.eval()
#     model.to(device)

#     pgd_attack = PGD(
#         model=model,
#         epsilon=epsilon,
#         alpha=alpha,
#         iterations=iterations,
#         random_start=random_start,
#         targeted=False,
#         device=device
#     )

#     test_loader = DataLoader(
#         test_dataset,
#         batch_size=batch_size,
#         shuffle=False
#     )

#     adv_images = []
#     adv_labels = []

#     for images, labels in tqdm(test_loader, desc="Generating PGD test set"):
#         images = images.to(device)
#         labels = labels.to(device)

#         with torch.enable_grad():
#             images_adv = pgd_attack.attack(images, labels)

#         adv_images.append(images_adv.cpu())
#         adv_labels.append(labels.cpu())

#     adv_images = torch.cat(adv_images)
#     adv_labels = torch.cat(adv_labels)

#     adv_test_dataset = TensorDataset(adv_images, adv_labels)

#     return adv_test_dataset



# def generate_and_save_pgd_test(
#     model,
#     test_dataset,
#     device,
#     save_path="datasets"
# ):
#     adv_test_dataset = create_pgd_test_dataset(
#         model=model,
#         test_dataset=test_dataset,
#         device=device,
#         epsilon=0.02,
#         alpha=0.004,
#         iterations=5
#     )

#     torch.save(
#         adv_test_dataset,
#         os.path.join(save_path, "test_pgd_eps002.pt")
#     )

#     print("Saved PGD adversarial test set")


# Clean Train and Test Datasets Creation

In [ ]:
import os
import shutil
import random
from pathlib import Path

In [3]:
def create_and_save_datasets(
    data_path,
    save_path,
    train_ratio=0.8,
    seed=42
):
    random.seed(seed)

    data_path = Path(data_path)
    save_path = Path(save_path)

    train_dir = save_path / "train_clean"
    test_dir = save_path / "test_clean"

    train_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    classes = [d for d in data_path.iterdir() if d.is_dir()]

    print(f"Found {len(classes)} classes")

    for class_dir in classes:
        class_name = class_dir.name

        # Créer les dossiers de sortie
        (train_dir / class_name).mkdir(exist_ok=True)
        (test_dir / class_name).mkdir(exist_ok=True)

        # Lister les images
        images = list(class_dir.glob("*"))
        images = [img for img in images if img.suffix.lower() in [".jpg", ".png", ".jpeg"]]

        random.shuffle(images)

        n_train = int(len(images) * train_ratio)

        train_images = images[:n_train]
        test_images = images[n_train:]

        # Copier les fichiers
        for img_path in train_images:
            shutil.copy(img_path, train_dir / class_name / img_path.name)

        for img_path in test_images:
            shutil.copy(img_path, test_dir / class_name / img_path.name)

        print(
            f"Class {class_name}: "
            f"{len(train_images)} train / {len(test_images)} test"
        )

    print("\nDataset split completed.")
    print(f"Train directory: {train_dir}")
    print(f"Test directory: {test_dir}")


create_and_save_datasets(data_path="data/EuroSAT_RGB", save_path="datasets/EuroSAT_RGB", train_ratio=0.8, seed=42)

Found 10 classes
Class River: 2000 train / 500 test
Class Pasture: 1600 train / 400 test
Class AnnualCrop: 2400 train / 600 test
Class HerbaceousVegetation: 2400 train / 600 test
Class SeaLake: 2400 train / 600 test
Class Forest: 2400 train / 600 test
Class Residential: 2400 train / 600 test
Class Highway: 2000 train / 500 test
Class PermanentCrop: 2000 train / 500 test
Class Industrial: 2000 train / 500 test

Dataset split completed.
Train directory: datasets/EuroSAT_RGB/train_clean
Test directory: datasets/EuroSAT_RGB/test_clean


# Baseline Model Creation

In [1]:
import sys

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--train",
    "--evaluate",
    "--visualize",
    "--epochs", "50",
    "--patience", "20",
    "--lr", "0.001",
    "--batch-size", "32",
    "--seed", "42",
    "--data-path-train", "datasets/EuroSAT_RGB/train_clean",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_clean",
    "--save-model-path", "output/models/baseline",
    "--save-plots-path", "outputs/plots/baseline_clean",
]

from main import main
main()

Using device: cuda
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_clean

Loading dataset...
Loaded 21600 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train samples: 15120
Validation samples: 6480
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400

Creating resnet18 model...
Model parameters: 11,173,962

Training resnet18 for 50 epochs...

Epoch 1/50
--------------------------------------------------


Train Loss: 1.1586 | Train Acc: 59.76%
Val Loss: 0.8792 | Val Acc: 69.74%
Saved best model with val_acc: 69.74%

Epoch 2/50
--------------------------------------------------


Train Loss: 0.8106 | Train Acc: 71.36%
Val Loss: 0.6420 | Val Acc: 76.37%
Saved best model with val_acc: 76.37%

Epoch 3/50
--------------------------------------------------


Train Loss: 0.6688 | Train Acc: 76.67%
Val Loss: 0.5362 | Val Acc: 81.57%
Saved best model with val_acc: 81.57%

Epoch 4/50
--------------------------------------------------


Train Loss: 0.5768 | Train Acc: 79.96%
Val Loss: 0.6550 | Val Acc: 77.79%

Epoch 5/50
--------------------------------------------------


...

Train Loss: 0.0171 | Train Acc: 99.46%
Val Loss: 0.0964 | Val Acc: 97.19%

Epoch 47/50
--------------------------------------------------


Train Loss: 0.0191 | Train Acc: 99.37%
Val Loss: 0.0861 | Val Acc: 97.39%

Epoch 48/50
--------------------------------------------------


Train Loss: 0.0154 | Train Acc: 99.43%
Val Loss: 0.0857 | Val Acc: 97.55%

Epoch 49/50
--------------------------------------------------


Train Loss: 0.0159 | Train Acc: 99.46%
Val Loss: 0.0929 | Val Acc: 97.39%

Epoch 50/50
--------------------------------------------------


Train Loss: 0.0138 | Train Acc: 99.55%
Val Loss: 0.0898 | Val Acc: 97.52%

Training completed! Best validation accuracy: 97.56%
Training history plot saved to: outputs/plots/resnet18_training_history.png
Training history plot saved to: outputs/plots/resnet18_training_history.png
Training metrics saved to CSV: outputs/plots/resnet18_training_metrics.csv
Comprehensive training report saved to: outputs/plots/resnet18_training_report.txt

Evaluating model on test set...
Loaded best model for evaluation
Test Accuracy: 97.57%
Confusion matrix saved to: outputs/plots/confusion_matrix_normalized.png

Classification Report:
                      precision    recall  f1-score   support

          AnnualCrop       0.97      0.96      0.96       600
              Forest       0.99      0.99      0.99       600
HerbaceousVegetation       0.97      0.96      0.96       600
             Highway       0.97      0.99      0.98       500
          Industrial       0.97      0.99      0.98       500
    

# Adversarial Test Dataset Creation

In [1]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision.utils import save_image
from tqdm import tqdm
import json

from models import ResNet18
from data_loader.dataset import EuroSatDataset
from attacks.pgd import PGD
from config import BATCH_SIZE, DEVICE, MEAN, STD, SEED

In [2]:
def load_model_and_create_attacked_test_dataset(
    model_path='outputs/models/best_model.pth',
    test_clean_path='datasets/EuroSAT_RGB/test_clean',
    save_adv_path='datasets/EuroSAT_RGB/test_pgd_eps002',
    epsilon_pixel=0.02,
    alpha_pixel=0.004,
    iterations=5,
    std=STD
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load model
    model = ResNet18().to(device)

    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            model.load_state_dict(checkpoint)
        print(f"Model loaded from {model_path}")
    else:
        raise FileNotFoundError(f"Model not found at {model_path}")

    model.eval()

    # Load clean test dataset
    dataset = EuroSatDataset(
        root_dir=test_clean_path,
        train=False
    )

    class_names = dataset.classes

    test_loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2
    )

    # Prepare save folders
    os.makedirs(save_adv_path, exist_ok=True)
    for cls in class_names:
        os.makedirs(os.path.join(save_adv_path, cls), exist_ok=True)

    # PGD attack
    epsilon = torch.tensor([epsilon_pixel / s for s in STD]).view(1,3,1,1)
    alpha   = torch.tensor([alpha_pixel / s for s in STD]).view(1,3,1,1)

    pgd = PGD(
        model=model,
        epsilon=epsilon,
        alpha=alpha,
        iterations=iterations,
        random_start=True,
        device=device,
        seed=SEED,
    )

    # Generate & save adversarial images
    img_idx = 0

    for images, labels in tqdm(test_loader, desc="Generating adversarial test set"):
        images = images.to(device)
        labels = labels.to(device)

        with torch.enable_grad():
            adv_images = pgd.attack(images, labels)

        for i in range(adv_images.size(0)):
            label = labels[i].item()
            class_name = class_names[label]

            save_path = os.path.join(
                save_adv_path,
                class_name,
                f"img_{img_idx}.png"
            )

            save_image(adv_images[i], save_path)
            img_idx += 1

    print(f"\nAdversarial test dataset saved to: {save_adv_path}")

    
    attack_config = {
        "attack": "PGD",
        "epsilon": epsilon.tolist() if torch.is_tensor(epsilon) else epsilon,
        "alpha": alpha.tolist() if torch.is_tensor(alpha) else alpha,
        "iterations": iterations,
        "random_start": True,
        "seed": SEED,
        "normalization": {
            "mean": MEAN,
            "std": STD
        }
    }

    os.makedirs("attacks/configs", exist_ok=True)
    with open(os.path.join("attacks/configs", "attack_config.json"), "w") as f:
        json.dump(attack_config, f, indent=4)

    return None

load_model_and_create_attacked_test_dataset(
    model_path='outputs/models/baseline/best_model.pth',
    test_clean_path='datasets/EuroSAT_RGB/test_clean',
    save_adv_path='datasets/EuroSAT_RGB/test_pgd_eps002',
    epsilon_pixel=0.02,
    alpha_pixel=0.004,
    iterations=10,
    std=STD
)

Using device: cuda
Model loaded from outputs/models/baseline/best_model.pth
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']

         [[0.0179]],

         [[0.0178]]]]) might be too large for ε=tensor([[[[0.0873]],

         [[0.0893]],

         [[0.0889]]]]), iterations=10


Generating adversarial test set: 100%|██████████| 169/169 [03:25<00:00,  1.22s/it]


Adversarial test dataset saved to: datasets/EuroSAT_RGB/test_pgd_eps002


In [1]:
import sys

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--evaluate",
    "--visualize",
    "--seed", "42",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_pgd_eps002",
    "--save-plots-path", "outputs/plots/baseline_pgd_eps002",
    "--save-model-path", "outputs/model/baseline",
]

from main import main
main()

Using device: cuda
Data paths: train = datasets/EuroSAT_RGB/train_clean, eval = datasets/EuroSAT_RGB/test_pgd_eps002

Loading dataset...
Loaded 21600 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train samples: 15120
Validation samples: 6480
Loaded 5400 images from 10 classes
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Test samples: 5400

Creating resnet18 model...
Model parameters: 11,173,962

Evaluating model on test set...
Test Accuracy: 11.20%
Confusion matrix saved to: outputs/plots/baseline_pgd_eps002/confusion_matrix_normalized.png

Classification Report:
                      precision    recall  f1-score   support

          AnnualCrop       0.00      0.00      0.00       600
              Forest       0.00      0.00      0.00       600
HerbaceousVegetatio

/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Predictions visualization saved to: outputs/plots/baseline_pgd_eps002/sample_predictions.png
Batch accuracy on 16 samples: 0.00%
Dataset samples plot saved to: outputs/plots/dataset_samples.png

Done!


# Adversarial and Noisy Train Datasets Creation

# Mixed Train Dataset Creation